In [242]:
import pandas as pd
import numpy as np

train_files = [
    "data/CRMLSSold202507.csv",
    "data/CRMLSSold202508.csv",
    "data/CRMLSSold202509.csv",
    "data/CRMLSSold202510.csv",
    "data/CRMLSSold202511.csv",
    "data/CRMLSSold202512.csv",
    "data/CRMLSSold202601.csv",
    "data/CRMLSSold202602.csv",
    "data/CRMLSSold202603.csv",
    "data/CRMLSSold202604.csv",
    "data/CRMLSSold202605.csv"
]
test_file = "data/CRMLSSold202606.csv"


# Load training set
df_train = pd.concat([pd.read_csv(f, low_memory=False) for f in train_files], ignore_index=True)

df_train = df_train[
    (df_train["PropertyType"] == "Residential") &
    (df_train["PropertySubType"] == "SingleFamilyResidence")
]
df_train = df_train[df_train["ClosePrice"] > 0] # Remove invalid ClosePrice
upper_limit = df_train["ClosePrice"].quantile(0.995)
df_train = df_train[df_train["ClosePrice"] < upper_limit] # Remove extreme ClosePrice
print(df_train.shape)
#print(df_train.info())


# Load test set
df_test = pd.read_csv(test_file)

df_test = df_test[
    (df_test["PropertyType"] == "Residential") &
    (df_test["PropertySubType"] == "SingleFamilyResidence")
]
df_test = df_test[df_test["ClosePrice"] > 0] # Remove invalid ClosePrice
df_test = df_test[df_test["ClosePrice"] < upper_limit] # Remove extreme ClosePrice
print(df_test.shape)
#print(df_test.info())

(117914, 78)
(12789, 78)


Handle Missing Values

In [243]:
key_cols = ["ClosePrice", "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet"]

summary = pd.DataFrame({
    "Data Type": df_train[key_cols].dtypes,
    "Missing Count": df_train[key_cols].isnull().sum(),
    "Missing %": (df_train[key_cols].isnull().mean() * 100).round(2)
})

summary_test = pd.DataFrame({
    "Data Type": df_test[key_cols].dtypes,
    "Missing Count": df_test[key_cols].isnull().sum(),
    "Missing %": (df_test[key_cols].isnull().mean() * 100).round(2)
})

print("Summary for training:", summary)
print("\nSummary for testing:", summary_test)

Summary for training:                       Data Type  Missing Count  Missing %
ClosePrice              float64              0       0.00
LivingArea              float64             44       0.04
BedroomsTotal           float64              0       0.00
BathroomsTotalInteger   float64              8       0.01
LotSizeSquareFeet       float64           2030       1.72

Summary for testing:                       Data Type  Missing Count  Missing %
ClosePrice              float64              0       0.00
LivingArea              float64              6       0.05
BedroomsTotal           float64              0       0.00
BathroomsTotalInteger   float64              0       0.00
LotSizeSquareFeet       float64            212       1.66


- Impute the missing values with the median computed from the training set for LivingArea, BathroomsTotalInteger, and LotSizeSquareFeet, since missing rate for each is less than 5%.
- Also flag missing for LotSizeSquareFeet.

In [244]:
# For Training Data
cleaned_train = df_train.copy()

# Flag missing for LotSizeSquareFeet
cleaned_train["Missing_LotSizeSquareFeet"] = cleaned_train["LotSizeSquareFeet"].isnull().astype(int)

# Impute missing values
for col in ["LivingArea", "BathroomsTotalInteger", "LotSizeSquareFeet"]:
    median_val = cleaned_train[col].median()
    cleaned_train[col] = cleaned_train[col].fillna(median_val)


# For Test Data
cleaned_test = df_test.copy()

# Flag missing for LotSizeSquareFeet
cleaned_test["Missing_LotSizeSquareFeet"] = cleaned_test["LotSizeSquareFeet"].isnull().astype(int)

# Impute missing values
for col in ["LivingArea",  "BathroomsTotalInteger", "LotSizeSquareFeet"]:
    median_val = cleaned_train[col].median() # Use median from training data
    cleaned_test[col] = cleaned_test[col].fillna(median_val)


Convert Categorical Fields

In [245]:
non_numeric_cols = cleaned_train.select_dtypes(exclude="number").columns
print(cleaned_train[non_numeric_cols].dtypes)

BuyerAgentAOR                  str
ListAgentAOR                   str
Flooring                       str
ViewYN                      object
WaterfrontYN                object
BasementYN                  object
PoolPrivateYN               object
ListAgentEmail                 str
CloseDate                      str
ListAgentFirstName             str
ListAgentLastName              str
UnparsedAddress                str
PropertyType                   str
ListOfficeName                 str
BuyerOfficeName                str
CoListOfficeName               str
ListAgentFullName              str
CoListAgentFirstName           str
CoListAgentLastName            str
BuyerAgentMlsId                str
BuyerAgentFirstName            str
BuyerAgentLastName             str
AssociationFeeFrequency        str
MLSAreaMajor                   str
CountyOrParish                 str
MlsStatus                      str
ElementarySchool               str
AttachedGarageYN            object
BuilderName         

1. Convert CloseDate, ContractStatusChangeDate , PurchaseContractDate, ListingContractDate to datetime 

In [246]:
date_cols = ["CloseDate", "ContractStatusChangeDate" , "PurchaseContractDate", "ListingContractDate"]

for col in date_cols:
    cleaned_train[col] = pd.to_datetime(cleaned_train[col], errors="coerce")
    cleaned_test[col] = pd.to_datetime(cleaned_test[col], errors="coerce")

cleaned_train[date_cols].info()
cleaned_test[date_cols].info()

<class 'pandas.DataFrame'>
Index: 117914 entries, 3 to 235782
Data columns (total 4 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   CloseDate                 117914 non-null  datetime64[us]
 1   ContractStatusChangeDate  117914 non-null  datetime64[us]
 2   PurchaseContractDate      117908 non-null  datetime64[us]
 3   ListingContractDate       117914 non-null  datetime64[us]
dtypes: datetime64[us](4)
memory usage: 4.5 MB
<class 'pandas.DataFrame'>
Index: 12789 entries, 2 to 24504
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   CloseDate                 12789 non-null  datetime64[us]
 1   ContractStatusChangeDate  12789 non-null  datetime64[us]
 2   PurchaseContractDate      12788 non-null  datetime64[us]
 3   ListingContractDate       12789 non-null  datetime64[us]
dtypes: datetime6

In [247]:
# Extract Date Features 
cleaned_train["CloseMonth"] = cleaned_train["CloseDate"].dt.month
cleaned_train["CloseYear"] = cleaned_train["CloseDate"].dt.year
cleaned_train["CloseDay"] = cleaned_train["CloseDate"].dt.day

cleaned_test["CloseMonth"] = cleaned_test["CloseDate"].dt.month
cleaned_test["CloseYear"] = cleaned_test["CloseDate"].dt.year
cleaned_test["CloseDay"] = cleaned_test["CloseDate"].dt.day

2. Check and convert booleon columns

In [248]:
yn_cols = [col for col in cleaned_train.columns if col.endswith("YN")]
for col in yn_cols:
    print(f"\nTrain: {col}")
    print(cleaned_train[col].value_counts(dropna=False))
    print(f"\nTest: {col}")
    print(cleaned_test[col].value_counts(dropna=False))


Train: ViewYN
ViewYN
True     66166
False    41090
NaN      10658
Name: count, dtype: int64

Test: ViewYN
ViewYN
True     7224
False    4321
NaN      1244
Name: count, dtype: int64

Train: WaterfrontYN
WaterfrontYN
NaN     117861
True        53
Name: count, dtype: int64

Test: WaterfrontYN
WaterfrontYN
NaN     12778
True       11
Name: count, dtype: int64

Train: BasementYN
BasementYN
NaN     115176
True      2738
Name: count, dtype: int64

Test: BasementYN
BasementYN
NaN     12475
True      314
Name: count, dtype: int64

Train: PoolPrivateYN
PoolPrivateYN
False    90872
True     18083
NaN       8959
Name: count, dtype: int64

Test: PoolPrivateYN
PoolPrivateYN
False    9732
True     2018
NaN      1039
Name: count, dtype: int64

Train: AttachedGarageYN
AttachedGarageYN
True     86556
False    17116
NaN      14242
Name: count, dtype: int64

Test: AttachedGarageYN
AttachedGarageYN
True     9475
False    1796
NaN      1518
Name: count, dtype: int64

Train: FireplaceYN
FireplaceYN
True    

In [249]:
for col in yn_cols:
    cleaned_train[col] = cleaned_train[col].fillna(False).astype(int)
    cleaned_test[col] = cleaned_test[col].fillna(False).astype(int)

Add new sample features for Week 6

In [250]:
cleaned_train["YearBuilt"].info()

<class 'pandas.Series'>
Index: 117914 entries, 3 to 235782
Series name: YearBuilt
Non-Null Count   Dtype  
--------------   -----  
117856 non-null  float64
dtypes: float64(1)
memory usage: 1.8 MB


In [251]:
# Add Property Age
cleaned_train["Missing_YearBuilt"] = cleaned_train["YearBuilt"].isna().astype(int)
cleaned_test["Missing_YearBuilt"] = cleaned_test["YearBuilt"].isna().astype(int)

CURRENT_YEAR = 2026
cleaned_train["PropertyAge"] = CURRENT_YEAR - cleaned_train["YearBuilt"]
cleaned_test["PropertyAge"] = CURRENT_YEAR - cleaned_test["YearBuilt"]

# Replace missing PropertyAge with median value
median_year = cleaned_train["PropertyAge"].median()
cleaned_train["PropertyAge"] = cleaned_train["PropertyAge"].fillna(median_year)
cleaned_test["PropertyAge"] = cleaned_test["PropertyAge"].fillna(median_year)

# Ensure PropertyAge is numeric
cleaned_train["PropertyAge"] = pd.to_numeric(
    cleaned_train["PropertyAge"],
    errors="coerce"
)
cleaned_test["PropertyAge"] = pd.to_numeric(
    cleaned_test["PropertyAge"],
    errors="coerce"
)

# Add Bed-Bath Ratio
cleaned_train["BedBathRatio"] = (
    cleaned_train["BedroomsTotal"] /
    cleaned_train["BathroomsTotalInteger"].replace(0, np.nan)
    )
cleaned_train["BedBathRatio"] = cleaned_train["BedBathRatio"].fillna(0)

cleaned_test["BedBathRatio"] = (
    cleaned_test["BedroomsTotal"] /
    cleaned_test["BathroomsTotalInteger"].replace(0, np.nan)
    )
cleaned_test["BedBathRatio"] = cleaned_test["BedBathRatio"].fillna(0)

In [252]:
import geopandas as gpd

# Read the downloaded school district GeoJSON into a GeoDataFrame
school_districts = gpd.read_file("data/DistrictAreas2526_-284845464123469011.geojson")

# Filter dataset to only include DistrictType == "Unified"
unified_districts = school_districts[school_districts["DistrictType"] == "Unified"]

# Convert each property's Latitude and Longtitude into a geographic point
train_gdf = gpd.GeoDataFrame(
    cleaned_train,
    geometry=gpd.points_from_xy(
        cleaned_train["Longitude"],
        cleaned_train["Latitude"]
    ),
    crs="EPSG:4326"
)

test_gdf = gpd.GeoDataFrame(
    cleaned_test,
    geometry=gpd.points_from_xy(
        cleaned_test["Longitude"],
        cleaned_test["Latitude"]
    ),
    crs="EPSG:4326"
)

unified_districts = unified_districts.to_crs(train_gdf.crs)

# Perform a spatial join
train_gdf = gpd.sjoin(
    train_gdf,
    unified_districts,
    how="left",
    predicate="within"
)

test_gdf = gpd.sjoin(
    test_gdf,
    unified_districts,
    how="left",
    predicate="within"
)

cleaned_train["SchoolDistrict"] = train_gdf["DistrictName"]
cleaned_test["SchoolDistrict"] = test_gdf["DistrictName"]

# Replace missing SchoolDistrict as "Unknown"
cleaned_train["SchoolDistrict"] = cleaned_train["SchoolDistrict"].fillna("Unknown")
cleaned_test["SchoolDistrict"] = cleaned_test["SchoolDistrict"].fillna("Unknown")

3.  Perform One-Hot Encoding on Selected Categotical Columns

In [253]:
categorical_cols = ["City", "PostalCode", "CountyOrParish", "SchoolDistrict"]
print(cleaned_train[categorical_cols].head())

print("\nNumber of unique values")
for col in categorical_cols:
    print(f"{col}: ", cleaned_train[col].nunique())

            City PostalCode CountyOrParish              SchoolDistrict
3          Vista      92081      San Diego               Vista Unified
4   Palm Springs      92264      Riverside        Palm Springs Unified
8       Monterey      93940       Monterey  Monterey Peninsula Unified
9      Escondido      92026      San Diego                     Unknown
10     Los Altos      94022    Santa Clara                     Unknown

Number of unique values
City:  995
PostalCode:  2007
CountyOrParish:  60
SchoolDistrict:  322


- Only one-hot encode on CountyOrParish, top 200 City and PostalCode, and SchoolDistrict

In [254]:
top_cities = cleaned_train["City"].value_counts().nlargest(200).index
top_postal = cleaned_train["PostalCode"].value_counts().nlargest(200).index

# Replace cities and postal codes not in top 200 with "other"
# On training set
cleaned_train["City_grouped"] = cleaned_train["City"].apply(
    lambda x: x if x in top_cities else "Other"
)
cleaned_train["PostalCode_grouped"] = cleaned_train["PostalCode"].apply(
    lambda x: x if x in top_postal else "Other"
)

# On test set
cleaned_test["City_grouped"] = cleaned_test["City"].apply(
    lambda x: x if x in top_cities else "Other"
)
cleaned_test["PostalCode_grouped"] = cleaned_test["PostalCode"].apply(
    lambda x: x if x in top_postal else "Other"
)

categorical_cols = ["City_grouped", "PostalCode_grouped", "CountyOrParish", "SchoolDistrict"]
cleaned_train = cleaned_train.drop(columns=["City", "PostalCode"])
cleaned_test = cleaned_test.drop(columns=["City", "PostalCode"])

In [255]:
cleaned_train.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,Missing_LotSizeSquareFeet,CloseMonth,CloseYear,CloseDay,Missing_YearBuilt,PropertyAge,BedBathRatio,SchoolDistrict,City_grouped,PostalCode_grouped
3,SanDiego,SanDiego,NaN,0,0,0,0,1040000.0,1120247618,agentsmylie@gmail.com,...,0,7,2025,31,0,38.0,1.000000,Vista Unified,Vista,Other
4,PalmSprings,PalmSprings,"Carpet,Tile",1,0,0,1,2400000.0,1120245509,reagan.richter@compass.com,...,0,7,2025,31,0,72.0,1.000000,Palm Springs Unified,Palm Springs,92264
8,Mlslistings,Mlslistings,"Carpet,Laminate,Stone,Tile",1,0,0,0,NaN,1120239726,ed@carmelrealtycompany.com,...,0,7,2025,23,0,58.0,1.333333,Monterey Peninsula Unified,Other,Other
9,PacificSouthwest,PacificSouthwest,NaN,1,0,0,0,1050000.0,1120231282,jared@clarkerealtors.com,...,0,7,2025,29,0,78.0,1.000000,Unknown,Escondido,92026
10,Mlslistings,Mlslistings,NaN,0,0,0,0,NaN,1120229958,miranda.bothe@gmail.com,...,0,7,2025,31,0,26.0,1.000000,Unknown,Other,Other


In [256]:
# One-hot encoding
from sklearn.preprocessing import OneHotEncoder

onehotencoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)

train_encoded = onehotencoder.fit_transform(cleaned_train[categorical_cols])
test_encoded = onehotencoder.transform(cleaned_test[categorical_cols])

In [257]:

from scipy import sparse

X_train = sparse.hstack([cleaned_train.drop(columns=categorical_cols).select_dtypes(include="number").values, train_encoded])
X_test = sparse.hstack([cleaned_test.drop(columns=categorical_cols).select_dtypes(include="number").values, test_encoded])

# Convert into dataframe
feature_names = list(cleaned_train.drop(columns=categorical_cols).select_dtypes(include="number").columns) + list(onehotencoder.get_feature_names_out(categorical_cols))

X_train_df = pd.DataFrame(
    X_train.toarray(),
    columns = feature_names
)
X_test_df = pd.DataFrame(
    X_test.toarray(),
    columns = feature_names
)

print(X_train_df["ViewYN"].unique())

#X_train_df = pd.DataFrame.sparse.from_spmatrix(X_train, columns=feature_names)
#X_test_df = pd.DataFrame.sparse.from_spmatrix(X_test, columns=feature_names)

print("\nTraining set shape: ", X_train_df.shape)
print("Test set shape: ", X_test_df.shape)


[0. 1.]

Training set shape:  (117914, 828)
Test set shape:  (12789, 828)


Save Cleaned CSV

In [258]:
X_train_df.to_csv("data/cleaned_train.csv", index=False)
X_test_df.to_csv("data/cleaned_test.csv", index=False)

In [259]:
X_train_df.columns.to_list()

['ViewYN',
 'WaterfrontYN',
 'BasementYN',
 'PoolPrivateYN',
 'OriginalListPrice',
 'ListingKey',
 'ClosePrice',
 'Latitude',
 'Longitude',
 'LivingArea',
 'ListPrice',
 'DaysOnMarket',
 'FireplacesTotal',
 'AboveGradeFinishedArea',
 'ListingKeyNumeric',
 'TaxAnnualAmount',
 'AttachedGarageYN',
 'ParkingTotal',
 'LotSizeAcres',
 'YearBuilt',
 'StreetNumberNumeric',
 'BathroomsTotalInteger',
 'TaxYear',
 'BuildingAreaTotal',
 'BedroomsTotal',
 'ElementarySchoolDistrict',
 'BelowGradeFinishedArea',
 'CoveredSpaces',
 'FireplaceYN',
 'Stories',
 'LotSizeArea',
 'MainLevelBedrooms',
 'NewConstructionYN',
 'GarageSpaces',
 'AssociationFee',
 'LotSizeSquareFeet',
 'MiddleOrJuniorSchoolDistrict',
 'Missing_LotSizeSquareFeet',
 'CloseMonth',
 'CloseYear',
 'CloseDay',
 'Missing_YearBuilt',
 'PropertyAge',
 'BedBathRatio',
 'City_grouped_29 Palms',
 'City_grouped_Adelanto',
 'City_grouped_Alameda',
 'City_grouped_Anaheim',
 'City_grouped_Anaheim Hills',
 'City_grouped_Antioch',
 'City_grouped_A

In [260]:
X_train_df["PropertyAge"].info()

<class 'pandas.Series'>
RangeIndex: 117914 entries, 0 to 117913
Series name: PropertyAge
Non-Null Count   Dtype  
--------------   -----  
117914 non-null  float64
dtypes: float64(1)
memory usage: 921.3 KB
